In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive/Thesis")
EXP_DIR = BASE_DIR / "Structural_MorphBPE_Experiment"

DATA_DIR = EXP_DIR / "data"
MODEL_DIR = EXP_DIR / "models"
RESULT_DIR = EXP_DIR / "results"
LOG_DIR = EXP_DIR / "logs"

for directory in [DATA_DIR, MODEL_DIR, RESULT_DIR, LOG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Experiment directory:", EXP_DIR)

Experiment directory: /content/drive/MyDrive/Thesis/Structural_MorphBPE_Experiment


In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import unicodedata
from collections import Counter, defaultdict

In [4]:
DATA_DIR = Path("/content/drive/MyDrive/Thesis/Structural_MorphBPE_Experiment")

TRAIN_FILE = DATA_DIR / "data/igbo_train_corpus.txt"
LEXICON_FILE = DATA_DIR / "data/Fully_Validated_IGBO_Lexicon.txt"

print("Training corpus:", TRAIN_FILE)
print("Lexicon:", LEXICON_FILE)

print("Training corpus exists:", TRAIN_FILE.exists())
print("Lexicon exists:", LEXICON_FILE.exists())

Training corpus: /content/drive/MyDrive/Thesis/Structural_MorphBPE_Experiment/data/igbo_train_corpus.txt
Lexicon: /content/drive/MyDrive/Thesis/Structural_MorphBPE_Experiment/data/Fully_Validated_IGBO_Lexicon.txt
Training corpus exists: True
Lexicon exists: True


In [5]:
final_df = pd.read_csv(
    LEXICON_FILE,
    sep="\t",
    header=None,
    names=["word", "segmentation"],
    dtype=str,
    encoding="utf-8"
)

final_df["word"] = final_df["word"].str.strip()
final_df["segmentation"] = final_df["segmentation"].str.strip()

print("Lexicon rows:", len(final_df))
print("Unique words:", final_df["word"].nunique())
print("Missing words:", final_df["word"].isna().sum())
print("Missing segmentations:", final_df["segmentation"].isna().sum())

display(final_df.head(20))

Lexicon rows: 57998
Unique words: 48059
Missing words: 2
Missing segmentations: 2


,word,segmentation
gọọmentị,gọọmentị,"[['g', 'ọ', 'ọ', 'm', 'e', 'n', 't', 'ị']]"
ndị,ndị,"[['n', 'd', 'ị']]"
onwuemeodo,onwu + eme + odo,"[['o', 'n', 'w', 'u'], ['e', 'm', 'e'], ['o', ..."
onye,onye,"[['o', 'n', 'y', 'e']]"
aafọ,aafọ,"[['a', 'a', 'f', 'ọ']]"
aakụkọ,aakụkọ,"[['a', 'a', 'k', 'ụ', 'k', 'ọ']]"
aba,aba,"[['a', 'b', 'a']]"
ababeghị,a + ba + be + ghị,"[['a'], ['b', 'a'], ['b', 'e'], ['g', 'h', 'ị']]"
abacha,abacha,"[['a', 'b', 'a', 'c', 'h', 'a']]"
abagana,abagana,"[['a', 'b', 'a', 'g', 'a', 'n', 'a']]"


In [6]:
lexicon = (
    final_df
    .drop_duplicates(subset=["word", "segmentation"])
    .reset_index(drop=True)
)

print("Original lexicon rows:", len(final_df))
print("After exact deduplication:", len(lexicon))
print("Unique words:", lexicon["word"].nunique())

Original lexicon rows: 57998
After exact deduplication: 48060
Unique words: 48059


In [7]:
import pandas as pd
import ast
from pathlib import Path

LEXICON_FILE = DATA_DIR / "data/Fully_Validated_IGBO_Lexicon.txt"

lexicon = pd.read_csv(
    LEXICON_FILE,
    sep="\t",
    header=None,
    names=[
        "word",
        "segmentation",
        "morph_structure"
    ],
    dtype=str,
    encoding="utf-8"
)

print("Rows:", len(lexicon))
print("Columns:", lexicon.columns.tolist())

display(lexicon.head(10))

Rows: 57998
Columns: ['word', 'segmentation', 'morph_structure']


,word,segmentation,morph_structure
0,gọọmentị,gọọmentị,"[['g', 'ọ', 'ọ', 'm', 'e', 'n', 't', 'ị']]"
1,ndị,ndị,"[['n', 'd', 'ị']]"
2,onwuemeodo,onwu + eme + odo,"[['o', 'n', 'w', 'u'], ['e', 'm', 'e'], ['o', ..."
3,onye,onye,"[['o', 'n', 'y', 'e']]"
4,aafọ,aafọ,"[['a', 'a', 'f', 'ọ']]"
5,aakụkọ,aakụkọ,"[['a', 'a', 'k', 'ụ', 'k', 'ọ']]"
6,aba,aba,"[['a', 'b', 'a']]"
7,ababeghị,a + ba + be + ghị,"[['a'], ['b', 'a'], ['b', 'e'], ['g', 'h', 'ị']]"
8,abacha,abacha,"[['a', 'b', 'a', 'c', 'h', 'a']]"
9,abagana,abagana,"[['a', 'b', 'a', 'g', 'a', 'n', 'a']]"


In [8]:
def parse_morph_structure(value):
    if pd.isna(value):
        return None

    try:
        structure = ast.literal_eval(value)

        if isinstance(structure, list):
            return structure

        return None

    except (ValueError, SyntaxError):
        return None

In [9]:
lexicon["morph_structure"] = (
    lexicon["morph_structure"]
    .apply(parse_morph_structure)
)

In [10]:
print(
    "Valid morphological structures:",
    lexicon["morph_structure"].notna().sum()
)

print(
    "Missing morphological structures:",
    lexicon["morph_structure"].isna().sum()
)

Valid morphological structures: 57996
Missing morphological structures: 2


In [11]:
display(
    lexicon[
        ["word", "segmentation", "morph_structure"]
    ].head(20)
)

,word,segmentation,morph_structure
0,gọọmentị,gọọmentị,"[[g, ọ, ọ, m, e, n, t, ị]]"
1,ndị,ndị,"[[n, d, ị]]"
2,onwuemeodo,onwu + eme + odo,"[[o, n, w, u], [e, m, e], [o, d, o]]"
3,onye,onye,"[[o, n, y, e]]"
4,aafọ,aafọ,"[[a, a, f, ọ]]"
5,aakụkọ,aakụkọ,"[[a, a, k, ụ, k, ọ]]"
6,aba,aba,"[[a, b, a]]"
7,ababeghị,a + ba + be + ghị,"[[a], [b, a], [b, e], [g, h, ị]]"
8,abacha,abacha,"[[a, b, a, c, h, a]]"
9,abagana,abagana,"[[a, b, a, g, a, n, a]]"


In [12]:
for i in range(10):
    print(
        repr(lexicon.loc[i, "word"]),
        "→",
        repr(lexicon.loc[i, "morph_structure"])
    )

'gọọmentị' → [['g', 'ọ', 'ọ', 'm', 'e', 'n', 't', 'ị']]
'ndị' → [['n', 'd', 'ị']]
'onwuemeodo' → [['o', 'n', 'w', 'u'], ['e', 'm', 'e'], ['o', 'd', 'o']]
'onye' → [['o', 'n', 'y', 'e']]
'aafọ' → [['a', 'a', 'f', 'ọ']]
'aakụkọ' → [['a', 'a', 'k', 'ụ', 'k', 'ọ']]
'aba' → [['a', 'b', 'a']]
'ababeghị' → [['a'], ['b', 'a'], ['b', 'e'], ['g', 'h', 'ị']]
'abacha' → [['a', 'b', 'a', 'c', 'h', 'a']]
'abagana' → [['a', 'b', 'a', 'g', 'a', 'n', 'a']]


In [13]:
def flatten_morph_structure(structure):
    if not structure:
        return ""

    return "".join(
        "".join(unit)
        for unit in structure
    )

In [14]:
lexicon["reconstructed_word"] = (
    lexicon["morph_structure"]
    .apply(flatten_morph_structure)
)

In [15]:
import pandas as pd
import ast
from pathlib import Path

# Directory already defined
# DATA_DIR = Path(...)

LEXICON_FILE = DATA_DIR / "data/Fully_Validated_IGBO_Lexicon.txt"

lexicon = pd.read_csv(
    LEXICON_FILE,
    sep="\t",
    header=None,
    names=["word", "segmentation", "morph_structure"],
    dtype=str,
    encoding="utf-8"
)

print("Lexicon rows:", len(lexicon))
print("Columns:", lexicon.columns.tolist())

display(lexicon.head())

Lexicon rows: 57998
Columns: ['word', 'segmentation', 'morph_structure']


,word,segmentation,morph_structure
0,gọọmentị,gọọmentị,"[['g', 'ọ', 'ọ', 'm', 'e', 'n', 't', 'ị']]"
1,ndị,ndị,"[['n', 'd', 'ị']]"
2,onwuemeodo,onwu + eme + odo,"[['o', 'n', 'w', 'u'], ['e', 'm', 'e'], ['o', ..."
3,onye,onye,"[['o', 'n', 'y', 'e']]"
4,aafọ,aafọ,"[['a', 'a', 'f', 'ọ']]"


In [16]:
lexicon["morph_structure"] = lexicon["morph_structure"].apply(
    lambda x: ast.literal_eval(x) if pd.notna(x) else None
)

In [17]:
print(
    "Valid morphological structures:",
    lexicon["morph_structure"].notna().sum()
)

print(
    "Missing morphological structures:",
    lexicon["morph_structure"].isna().sum()
)

Valid morphological structures: 57996
Missing morphological structures: 2


In [18]:
morph_lookup = dict(
    zip(
        lexicon["word"],
        lexicon["morph_structure"]
    )
)

print("Morphology lookup entries:", len(morph_lookup))

Morphology lookup entries: 57949


In [19]:
for word in ["gọọmentị", "ndị", "onwuemeodo", "ababeghị"]:
    print(word, "→", morph_lookup.get(word))

gọọmentị → [['g', 'ọ', 'ọ', 'm', 'e', 'n', 't', 'ị']]
ndị → [['n', 'd', 'ị']]
onwuemeodo → [['o', 'n', 'w', 'u'], ['e', 'm', 'e'], ['o', 'd', 'o']]
ababeghị → [['a'], ['b', 'a'], ['b', 'e'], ['g', 'h', 'ị']]


In [20]:
TRAIN_FILE = DATA_DIR / "data/igbo_train_corpus.txt"

with open(
    TRAIN_FILE,
    "r",
    encoding="utf-8"
) as f:
    train_lines = f.readlines()

print("Number of lines:", len(train_lines))
print("First line:")
print(repr(train_lines[0]))

Number of lines: 35078
First line:
'Sowore Revolution: Ka ekwe si akụ maka ngagharịiwe e ji maka ya nwụchie Sowore BBC Igbo kwụ chịm iwetara gị ihe na-aga ka a na-ekwu okwu ngagharịiwe na mpaghara dị iche iche na Naịjirịa.\n'


In [21]:
train_tokens = []

for line in train_lines:
    train_tokens.extend(line.split())

print("Total corpus tokens:", len(train_tokens))
print("Unique corpus tokens:", len(set(train_tokens)))

Total corpus tokens: 833843
Unique corpus tokens: 54493


In [22]:
scoped_corpus = []

covered = 0
uncovered = 0

for token in train_tokens:

    structure = morph_lookup.get(token)

    if structure is not None:
        scoped_corpus.append(structure)
        covered += 1

    else:
        scoped_corpus.append([
            list(token)
        ])
        uncovered += 1

print("Covered occurrences:", covered)
print("Uncovered occurrences:", uncovered)

print(
    f"Coverage: {covered / len(train_tokens):.4%}"
)

Covered occurrences: 833735
Uncovered occurrences: 108
Coverage: 99.9870%


In [23]:
for i in range(30):
    print(
        train_tokens[i],
        "→",
        scoped_corpus[i]
    )

Sowore → [['S', 'o', 'w', 'o', 'r', 'e']]
Revolution: → [['R', 'e', 'v', 'o', 'l', 'u', 't', 'i', 'o', 'n', ' ', ':']]
Ka → [['k', 'a']]
ekwe → [['e', 'k', 'w', 'e']]
si → [['s', 'i']]
akụ → [['a', 'k', 'ụ']]
maka → [['m', 'a', 'k', 'a']]
ngagharịiwe → [['n', 'g', 'a', 'g', 'h', 'a', 'r', 'ị'], ['i', 'w', 'e']]
e → [['e']]
ji → [['j', 'i']]
maka → [['m', 'a', 'k', 'a']]
ya → [['y', 'a']]
nwụchie → [['n', 'w', 'ụ'], ['c', 'h', 'i'], ['e']]
Sowore → [['S', 'o', 'w', 'o', 'r', 'e']]
BBC → [['B', 'B', 'C']]
Igbo → [['i', 'g', 'b', 'o']]
kwụ → [['k', 'w', 'ụ']]
chịm → [['c', 'h', 'ị'], ['m']]
iwetara → [['i'], ['w', 'e'], ['t', 'a'], ['r', 'a']]
gị → [['g', 'ị']]
ihe → [['i', 'h', 'e']]
na-aga → [['n', 'a', '-', 'a'], ['g', 'a']]
ka → [['k', 'a']]
a → [['a']]
na-ekwu → [['n', 'a', '-', 'e'], ['k', 'w', 'u']]
okwu → [['o', 'k', 'w', 'u']]
ngagharịiwe → [['n', 'g', 'a', 'g', 'h', 'a', 'r', 'ị'], ['i', 'w', 'e']]
na → [['n', 'a']]
mpaghara → [['m', 'p', 'a', 'g', 'h', 'a', 'r', 'a']]
dị → [['d

In [24]:
import json

SCOPED_CORPUS_FILE = DATA_DIR / "igbo_morph_scoped_corpus.json"

with open(
    SCOPED_CORPUS_FILE,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        scoped_corpus,
        f,
        ensure_ascii=False
    )

print(
    "Saved:",
    SCOPED_CORPUS_FILE
)

Saved: /content/drive/MyDrive/Thesis/Structural_MorphBPE_Experiment/igbo_morph_scoped_corpus.json


In [25]:
LOOKUP_FILE = DATA_DIR / "data/igbo_morphology_lookup.json"

with open(
    LOOKUP_FILE,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        morph_lookup,
        f,
        ensure_ascii=False
    )

print(
    "Saved:",
    LOOKUP_FILE
)

Saved: /content/drive/MyDrive/Thesis/Structural_MorphBPE_Experiment/data/igbo_morphology_lookup.json
